In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -U google-genai streamlit Pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 121.6 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
ERROR: pip's 

In [4]:
%%writefile app.py
import streamlit as st
from google import genai
from google.genai import types
import PIL.Image
import io

# --- PAGE CONFIG ---
st.set_page_config(page_title="Colab Multimodal Chat", layout="centered")
st.title("🤖 Gemini 3 Multi-Modal Chat")

# Setup API Key
# For Colab, you can also use userdata.get('GOOGLE_API_KEY') if using Colab Secrets
api_key = st.text_input("Enter Gemini API Key", type="password")

if not api_key:
    st.warning("Please enter your API Key to continue.")
    st.stop()

client = genai.Client(api_key=api_key)

# Initialize Session State
if "messages" not in st.session_state:
    st.session_state.messages = []

# Sidebar for Image Upload
with st.sidebar:
    st.header("Image Input")
    uploaded_file = st.file_uploader("Upload an image...", type=["jpg", "jpeg", "png"])
    if uploaded_file:
        st.image(uploaded_file, caption="Preview", use_container_width=True)

# Display Chat History
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if "image" in message:
            st.image(message["image"])

# User Input Logic
if prompt := st.chat_input("Say something or type 'Generate: [prompt]'"):

    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        try:
            # Branch 1: Image Generation
            if prompt.lower().startswith("generate:"):
                gen_prompt = prompt[9:].strip()
                st.write(f"🎨 Creating: *{gen_prompt}*...")

                result = client.models.generate_images(
                    model='imagen-3',
                    prompt=gen_prompt,
                    config=types.GenerateImagesConfig(number_of_images=1)
                )

                img_bytes = result.generated_images[0].image.image_bytes
                st.image(img_bytes)
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": f"Generated: {gen_prompt}",
                    "image": img_bytes
                })

            # Branch 2: Image Understanding / Chat
            else:
                inputs = [prompt]
                if uploaded_file:
                    inputs.append(PIL.Image.open(uploaded_file))

                response = client.models.generate_content(
                    model="gemini-3-flash-preview",
                    contents=inputs
                )

                st.markdown(response.text)
                st.session_state.messages.append({"role": "assistant", "content": response.text})

        except Exception as e:
            st.error(f"An error occurred: {e}")

Writing app.py


In [5]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-05-03 05:03:42--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-05-03 05:03:43--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-03T05%3A50%3A07Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
import subprocess
import threading
import time

# 1. Start Streamlit in the background
def run_streamlit():
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

threading.Thread(target=run_streamlit).start()
time.sleep(3) # Give it a second to warm up

# 2. Expose the port via Cloudflare
# This will output a link ending in '.trycloudflare.com'
!cloudflared tunnel --url http://localhost:8501

2026-05-03T05:04:29Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-03T05:04:29Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-03T05:04:31Z INF +--------------------------------------------------------------------------------------------+
2026-05-03T05:04:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-03T05:04:31Z INF |  https://includes-been-varies-emission.trycloudflare.c